# gen-gec-errant Pipeline — Colab

Run the full **generate → GEC → ERRANT → analysis** pipeline for a single model.

**Data source:** `phd-experimental-data/cefr-classification/data/splits/`  
**Models:** `_p/artificial-learners/models/` (fine-tuned) or HuggingFace Hub (native)  
**Output:** `_p/artificial-learners/generation-gec-errant/results/`  

**Runtime:** Depends on model size and dataset limits. ~10–30 min per dataset on T4 GPU.

*Last updated: 2026-03-16*

## 0. Mount Google Drive & Install

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%time
# Install core dependencies
!pip install torch transformers errant spacy numpy scipy matplotlib pandas pyyaml accelerate -q
!python -m spacy download en_core_web_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.3/499.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 111.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
CPU times: user 1.29 s, sys: 194 ms, total: 1.48 s
Wall time: 20.1 s


In [ ]:
%%time
# Install gen_gec_errant from GitHub (public repo)
!pip install git+https://github.com/berstearns/public-automatic-generation-correction-errortagging-v2.git -q

## 1. Configuration

Select a model from the registry. Set `MODEL_KEY` to one of the available models below.

**Smoke test:** Set `SMOKE_TEST = True` to run only 5 sentences from CELVA-SP with native GPT-2.

In [ ]:
from pathlib import Path
from datetime import datetime
import torch

from gen_gec_errant.registry import MODEL_REGISTRY, get_datasets, build_pipeline_config, PathConfig
from gen_gec_errant.colab import resolve_model_path, cleanup_local_model

# ══════════════════════════════════════════════════════════════════════
#  USER CONFIG — edit these
# ══════════════════════════════════════════════════════════════════════
GDRIVE = Path('/content/drive/MyDrive')

DATA_ROOT    = GDRIVE / 'phd-experimental-data/cefr-classification/data/splits'
MODELS_ROOT  = GDRIVE / '_p/artificial-learners/models'
OUTPUT_ROOT  = GDRIVE / '_p/artificial-learners/generation-gec-errant/results'

SMOKE_TEST = True

if SMOKE_TEST:
    MODEL_KEY      = 'gpt2-small-native'
    MAX_SENTENCES  = 5
    DATASETS_TO_RUN = ['norm-CELVA-SP']
else:
    MODEL_KEY      = 'ft-gpt2-small'       # <-- change this
    MAX_SENTENCES  = None                   # None = all data
    DATASETS_TO_RUN = None                  # None = all datasets

INCLUDE_LEARNER_BASELINE = True
# ══════════════════════════════════════════════════════════════════════

# Derived
TIMESTAMP = datetime.now().strftime('%Y%m%d-%H%M%S')
paths = PathConfig(data_root=DATA_ROOT, models_root=MODELS_ROOT, output_root=OUTPUT_ROOT)
model = MODEL_REGISTRY[MODEL_KEY]
datasets = get_datasets(DATASETS_TO_RUN)
RUN_DIR = OUTPUT_ROOT / f"{MODEL_KEY}-{TIMESTAMP}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# ── Device ──
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'\nMode:        {"SMOKE TEST" if SMOKE_TEST else "FULL RUN"}')
print(f'Model:       {MODEL_KEY} ({model.params})')
print(f'Output dir:  {RUN_DIR}')
print(f'Data root:   {DATA_ROOT}')
print(f'Models root: {MODELS_ROOT}')
print(f'Max sent.:   {MAX_SENTENCES or "all"}')

## 1a. Copy Model to Local Disk (faster I/O)

Copying model weights from GDrive to Colab's local SSD avoids slow I/O during inference.

In [ ]:
LOCAL_MODEL_PATH = resolve_model_path(model, paths)
if LOCAL_MODEL_PATH is None:
    raise RuntimeError(f'Model not found on GDrive for {MODEL_KEY}')
print(f'Effective model path: {LOCAL_MODEL_PATH}')

## 2. Run Pipeline Per Dataset

Runs the full 5-stage pipeline (data loading → generation → GEC → ERRANT annotation → analysis) for each dataset.

In [ ]:
import logging
import time

logging.basicConfig(level=logging.INFO, format='%(name)s | %(message)s')

from gen_gec_errant.pipeline import run_pipeline

print(f'Available VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB' if device == 'cuda' else 'Running on CPU')

In [ ]:
%%time

all_summaries = {}
all_comparisons = {}
t_total = time.time()

for ds in datasets:
    ds_path = paths.dataset_path(ds)

    print('\n' + '=' * 70)
    print(f'  DATASET: {ds.name}')
    print(f'  {ds.description}')
    print('=' * 70)

    if not ds_path.exists():
        print(f'  SKIPPED: file not found at {ds_path}')
        continue

    output_dir = RUN_DIR / ds.name
    raw_results = output_dir / 'raw_results.json'

    if raw_results.exists():
        print(f'  Already completed (raw_results.json exists). Skipping.')
        continue

    config = build_pipeline_config(
        model, ds, paths,
        model_path=LOCAL_MODEL_PATH,
        max_sentences=MAX_SENTENCES,
        include_learner_baseline=INCLUDE_LEARNER_BASELINE,
        output_dir=str(output_dir),
    )
    print(f'  Model path: {LOCAL_MODEL_PATH}')

    t0 = time.time()
    summaries, comparison = run_pipeline(config)
    elapsed = time.time() - t0

    all_summaries[ds.name] = summaries
    all_comparisons[ds.name] = comparison

    print(f'\n  {ds.name} completed in {elapsed / 60:.1f} min')
    for s in summaries:
        ppl = s.get('ppl_mean', 'N/A')
        ppl_str = f'{ppl:.2f}' if isinstance(ppl, (int, float)) else str(ppl)
        avg_err = s.get('avg_errors_per_sentence', 'N/A')
        avg_str = f'{avg_err:.2f}' if isinstance(avg_err, (int, float)) else str(avg_err)
        print(f'    {s["model_name"]}: PPL={ppl_str}, Errors/sent={avg_str}')

    # Free GPU memory between datasets
    if device == 'cuda':
        torch.cuda.empty_cache()

total_elapsed = time.time() - t_total
print(f'\nAll datasets completed in {total_elapsed / 60:.1f} min')

## 3. Cross-Dataset Summary

In [ ]:
import json

cross_summary = {}

for ds in datasets:
    output_dir = RUN_DIR / ds.name
    entry = {'dataset': ds.name}

    model_summary_path = output_dir / f'{MODEL_KEY}_summary.json'
    baseline_summary_path = output_dir / 'learner_baseline_summary.json'

    if model_summary_path.exists():
        with open(model_summary_path) as f:
            data = json.load(f)
        entry['model'] = {
            'ppl_mean': data.get('ppl_mean'),
            'ppl_median': data.get('ppl_median'),
            'total_errors': data.get('total_errors'),
            'avg_errors_per_sentence': data.get('avg_errors_per_sentence'),
            'error_rate': data.get('error_rate'),
            'top_error_types': data.get('top_10_error_types', [])[:5],
        }

    if baseline_summary_path.exists():
        with open(baseline_summary_path) as f:
            data = json.load(f)
        entry['learner_baseline'] = {
            'total_errors': data.get('total_errors'),
            'avg_errors_per_sentence': data.get('avg_errors_per_sentence'),
            'error_rate': data.get('error_rate'),
            'top_error_types': data.get('top_10_error_types', [])[:5],
        }

    cross_summary[ds.name] = entry

# Print summary table
print(f'Cross-Dataset Summary for {MODEL_KEY} ({model.params})')
print(f'{"Dataset":<25} {"PPL Mean":>10} {"Errors/Sent":>12} {"Error Rate":>12}')
print(f'{"-"*25} {"-"*10} {"-"*12} {"-"*12}')

for name, data in cross_summary.items():
    m = data.get('model', {})
    ppl = m.get('ppl_mean', 'N/A')
    avg_err = m.get('avg_errors_per_sentence', 'N/A')
    err_rate = m.get('error_rate', 'N/A')
    ppl_s = f'{ppl:.2f}' if isinstance(ppl, (int, float)) else str(ppl)
    avg_s = f'{avg_err:.2f}' if isinstance(avg_err, (int, float)) else str(avg_err)
    rate_s = f'{err_rate:.3f}' if isinstance(err_rate, (int, float)) else str(err_rate)
    print(f'{name:<25} {ppl_s:>10} {avg_s:>12} {rate_s:>12}')

# Compare with learner baseline
print(f'\nLearner Baseline Comparison:')
print(f'{"Dataset":<25} {"Model Err/Sent":>14} {"Learner Err/Sent":>16} {"Ratio":>8}')
print(f'{"-"*25} {"-"*14} {"-"*16} {"-"*8}')

for name, data in cross_summary.items():
    m_err = data.get('model', {}).get('avg_errors_per_sentence')
    l_err = data.get('learner_baseline', {}).get('avg_errors_per_sentence')
    if m_err and l_err and l_err > 0:
        print(f'{name:<25} {m_err:>14.2f} {l_err:>16.2f} {m_err/l_err:>8.2f}')
    else:
        print(f'{name:<25} {"N/A":>14} {"N/A":>16} {"N/A":>8}')

## 4. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Error Rate Comparison (Model vs Learner Baseline) ──────────────────

ds_names = []
model_rates = []
learner_rates = []

ds_display = {
    'norm-CELVA-SP': 'CELVA-SP',
    'norm-EFCAMDAT-test': 'EFCAMDAT-test',
    'norm-KUPA-KEYS': 'KUPA-KEYS',
}

for name, data in cross_summary.items():
    m_err = data.get('model', {}).get('avg_errors_per_sentence')
    l_err = data.get('learner_baseline', {}).get('avg_errors_per_sentence')
    if m_err is not None and l_err is not None:
        ds_names.append(ds_display.get(name, name))
        model_rates.append(m_err)
        learner_rates.append(l_err)

if ds_names:
    x = np.arange(len(ds_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - width/2, model_rates, width, label=MODEL_KEY, color='#4C72B0')
    bars2 = ax.bar(x + width/2, learner_rates, width, label='Learner baseline', color='#DD8452')

    ax.set_ylabel('Avg errors per sentence')
    ax.set_title(f'Error Profile: {MODEL_KEY} vs Learner Baseline')
    ax.set_xticks(x)
    ax.set_xticklabels(ds_names)
    ax.legend()
    ax.bar_label(bars1, fmt='%.2f', padding=3)
    ax.bar_label(bars2, fmt='%.2f', padding=3)

    plt.tight_layout()
    fig.savefig(str(RUN_DIR / 'error_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {RUN_DIR / "error_comparison.png"}')
else:
    print('No results to plot.')

In [ ]:
# ── Top Error Types per Dataset ───────────────────────────────────────

n_plots = sum(1 for d in cross_summary.values() if 'model' in d and d['model'].get('top_error_types'))

if n_plots > 0:
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    plot_idx = 0
    for ds_name, data in cross_summary.items():
        model_data = data.get('model', {})
        top_types = model_data.get('top_error_types', [])
        if not top_types:
            continue

        labels = [t[0] if isinstance(t, (list, tuple)) else t.get('type', '?') for t in top_types]
        counts = [t[1] if isinstance(t, (list, tuple)) else t.get('count', 0) for t in top_types]

        ax = axes[plot_idx]
        ax.barh(labels[::-1], counts[::-1], color='#4C72B0')
        ax.set_xlabel('Count')
        ax.set_title(ds_display.get(ds_name, ds_name))
        plot_idx += 1

    fig.suptitle(f'Top Error Types: {MODEL_KEY}', fontsize=14, y=1.02)
    plt.tight_layout()
    fig.savefig(str(RUN_DIR / 'top_error_types.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {RUN_DIR / "top_error_types.png"}')
else:
    print('No error type data to plot.')

## 5. Save Everything to Google Drive

In [ ]:
import yaml

# Save cross-dataset summary
summary_path = RUN_DIR / 'cross_dataset_summary.json'
with open(summary_path, 'w') as f:
    json.dump(cross_summary, f, indent=2, default=str)
print(f'Cross-dataset summary: {summary_path}')

# Save run configuration
run_config = {
    'model_key': MODEL_KEY,
    'model_info': {
        'params': model.params,
        'model_family': model.model_family,
        'is_learner_tuned': model.is_learner_tuned,
        'description': model.description,
        'model_path': LOCAL_MODEL_PATH,
    },
    'pipeline': {
        'gec_model': 'grammarly/coedit-large',
        'gec_method': 'dedicated',
        'batch_size': model.batch_size,
        'gec_batch_size': model.gec_batch_size,
        'seed': 42,
        'max_sentences': MAX_SENTENCES,
    },
    'datasets': [ds.name for ds in datasets],
    'timestamp': TIMESTAMP,
    'device': device,
    'gpu': torch.cuda.get_device_name() if device == 'cuda' else None,
}

config_path = RUN_DIR / 'run_config.yaml'
with open(config_path, 'w') as f:
    yaml.dump(run_config, f, default_flow_style=False)
print(f'Run config: {config_path}')

# List all output files
print(f'\n=== All artifacts saved to {RUN_DIR} ===')
for p in sorted(RUN_DIR.rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        rel = p.relative_to(RUN_DIR)
        print(f'  {rel}  ({size_mb:.1f} MB)' if size_mb > 0.1 else f'  {rel}')